# Анализ центральности графа дружбы VK

**Задача:** для студентов учебной и параллельной групп построить граф дружеских связей во VK и определить:
- центральность по степени (количество связей);
- центральность по посредничеству (betweenness centrality);
- центральность по близости (closeness centrality);
- центральность собственного вектора (eigenvector centrality);
- студента с наибольшим количеством связей.

### Модель графа

- **Вершина** — студент из исходного файла.
- **Ребро** — два студента являются друзьями во VK.
- В граф включаются только люди из двух заданных групп. Друзья, которые не входят в список студентов, используются для проверки связи, но отдельными вершинами графа не становятся.

Это позволяет оценивать положение человека именно **внутри учебного сообщества**, а не среди всех его друзей во VK.

## 1. Установка библиотек

Запустите ячейку один раз. Для `.xlsx` нужен `openpyxl`.

In [3]:
%pip install -q pandas networkx requests matplotlib openpyxl

Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import re
import time
from getpass import getpass
from itertools import combinations

import pandas as pd
import networkx as nx
import requests
import matplotlib.pyplot as plt

## 2. Настройки

Положите файл с ID рядом с этим notebook и укажите его имя.

Поддерживаются `.xlsx`, `.xls`, `.csv`.

В таблице желательно иметь столбцы:
- `id` — ID пользователя VK;
- `group` — название группы, например `Основная` и `Параллельная`;
- `name` — имя студента (необязательно).

Если столбца `group` нет, вся выборка будет считаться одной группой.

In [5]:
INPUT_FILE = "data_VK.xlsx"

# Имя листа Excel. Для первого листа можно оставить 0.
EXCEL_SHEET = 0

# Версия VK API.
VK_API_VERSION = "5.199"

# Пауза между запросами, чтобы не создавать избыточную нагрузку.
REQUEST_DELAY = 0.35

# Количество попыток при временной ошибке.
MAX_RETRIES = 3

## 3. Загрузка списка студентов

Ячейка пытается самостоятельно найти подходящий столбец с ID.

In [6]:
def load_students(path):
    path = str(path)
    ext = os.path.splitext(path)[1].lower()

    if ext in [".xlsx", ".xls"]:
        df = pd.read_excel(path, sheet_name=EXCEL_SHEET)
    elif ext == ".csv":
        try:
            df = pd.read_csv(path)
        except UnicodeDecodeError:
            df = pd.read_csv(path, encoding="cp1251")
    else:
        raise ValueError("Поддерживаются только XLSX, XLS и CSV.")

    df.columns = [str(c).strip().lower() for c in df.columns]

    id_candidates = ["id", "vk_id", "user_id", "userid", "вк id", "vk"]
    id_col = next((c for c in id_candidates if c in df.columns), None)

    if id_col is None:
        best_col = None
        best_score = 0
        for col in df.columns:
            values = df[col].dropna().astype(str)
            if len(values) == 0:
                continue
            score = values.str.contains(
                r"(?:vk\.com/)?(?:id)?\d+|^\d+$",
                case=False,
                regex=True
            ).mean()
            if score > best_score:
                best_score = score
                best_col = col

        if best_col is None or best_score < 0.5:
            raise ValueError(
                "Не удалось определить столбец с VK ID. "
                "Переименуйте его в 'id'."
            )
        id_col = best_col

    result = pd.DataFrame()
    result["id_raw"] = df[id_col]

    if "name" in df.columns:
        result["name"] = df["name"].fillna("").astype(str).str.strip()
    elif "фио" in df.columns:
        result["name"] = df["фио"].fillna("").astype(str).str.strip()
    else:
        result["name"] = ""

    if "group" in df.columns:
        result["group"] = df["group"].fillna("Не указана").astype(str).str.strip()
    elif "группа" in df.columns:
        result["group"] = df["группа"].fillna("Не указана").astype(str).str.strip()
    else:
        result["group"] = "Не указана"

    def parse_vk_id(value):
        s = str(value).strip()
        # URL вида https://vk.com/id123 или vk.com/id123
        m = re.search(r"(?:vk\.com/)(?:id)?(\d+)", s, re.I)
        if m:
            return int(m.group(1))
        # Просто число
        m = re.fullmatch(r"(\d+)", s)
        if m:
            return int(m.group(1))
        # id123
        m = re.fullmatch(r"id(\d+)", s, re.I)
        if m:
            return int(m.group(1))
        # VK юзернейм: буквы, цифры, точки, подчёркивания, дефисы (мин. 2 символа)
        m = re.fullmatch(r"[a-zA-Z0-9][a-zA-Z0-9_.\-]+", s)
        if m:
            return s  # строка — будет разрешена через users.get
        return pd.NA

    result["id"] = result["id_raw"].map(parse_vk_id)
    result = result.dropna(subset=["id"]).copy()
    # Не кастуем в int64 сейчас — юзернеймы будут разрешены через API

    result = result.drop_duplicates(subset=["id"], keep="first").reset_index(drop=True)

    return result

students = load_students(INPUT_FILE)

print(f"Количество студентов в выборке: {len(students)}")
display(students.head(10))

## 4. Проверка выборки

In [7]:
print("Количество студентов по группам:")
display(students["group"].value_counts().rename_axis("group").to_frame("count"))

print("\nДубликаты ID:", students["id"].duplicated().sum())
print("Пропуски ID:", students["id"].isna().sum())

Количество студентов по группам:


,count
group,
BSBO-10-26,3
BSBO-12-26,1



Дубликаты ID: 0
Пропуски ID: 0


## 5. Токен VK API

Используйте собственный пользовательский access token. Не вставляйте его в публичный notebook и не отправляйте в репозиторий GitHub.

Код ниже хранит токен только в памяти текущей сессии Jupyter.

In [ ]:
VK_TOKEN = getpass("Введите VK access token: ").strip()

if not VK_TOKEN:
    raise ValueError("Токен не введён.")

[{'id': 1, 'first_name': 'Павел', 'last_name': 'Дуров', 'can_access_closed': True, 'is_closed': False}]


## 6. Вызов VK API

Используются методы `users.get` для профилей и `friends.get` для списка друзей.

In [18]:
API_URL = "https://api.vk.com/method/"

session = requests.Session()

def vk_request(method, **params):
    params["access_token"] = VK_TOKEN
    params["v"] = VK_API_VERSION

    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = session.get(
                API_URL + method,
                params=params,
                timeout=20
            )
            response.raise_for_status()
            data = response.json()

            if "error" in data:
                error = data["error"]
                code = error.get("error_code")
                msg = error.get("error_msg", "Неизвестная ошибка VK API")
                raise RuntimeError(f"VK API error {code}: {msg}")

            return data["response"]

        except Exception as exc:
            last_error = exc
            if attempt < MAX_RETRIES:
                time.sleep(attempt * 1.5)
            else:
                raise RuntimeError(
                    f"Не удалось выполнить запрос {method}: {last_error}"
                ) from last_error

## 7. Получение профилей студентов

Сохраняем только минимально необходимые сведения: ID, имя и фамилию. Дополнительные персональные поля не собираются.

In [21]:
student_identifiers = students["id"].tolist()  # числа и/или юзернеймы

profiles = []
identifier_order = []

for start in range(0, len(student_identifiers), 1000):
    batch = student_identifiers[start:start + 1000]
    resp = vk_request(
        "users.get",
        user_ids=",".join(map(str, batch))
    )
    profiles.extend(resp)
    identifier_order.extend(batch)
    time.sleep(REQUEST_DELAY)

profiles_df = pd.DataFrame(profiles)

if not profiles_df.empty:
    # Маппинг: оригинальный идентификатор (юзернейм или число) -> числовой id VK
    id_resolution = {orig: prof["id"] for orig, prof in zip(identifier_order, profiles)}

    students["id"] = students["id"].map(lambda x: id_resolution.get(x, x))
    students["id"] = pd.to_numeric(students["id"], errors="coerce")
    students = students.dropna(subset=["id"]).copy()
    students["id"] = students["id"].astype("int64")

    profiles_df["name_vk"] = (
        profiles_df["first_name"].fillna("") + " " +
        profiles_df["last_name"].fillna("")
    ).str.strip()

    if "deactivated" not in profiles_df.columns:
        profiles_df["deactivated"] = None

    students = students.drop(columns=["name"], errors="ignore").merge(
        profiles_df[["id", "name_vk", "deactivated"]],
        on="id",
        how="left"
    )

    students["name"] = students["name_vk"].fillna("")
    students = students.drop(columns=["name_vk"], errors="ignore")

else:
    students["name"] = students.get("name", "")
    students["deactivated"] = None

print(f"Загружено студентов: {len(students)}")
display(students.head())

## 8. Получение списков друзей

Для каждого студента запрашивается список его друзей. API может не вернуть список полностью или может отказать в доступе для конкретного профиля. Такие случаи фиксируются отдельно и не скрываются.

In [22]:
def get_all_friends(user_id, count=5000):
    all_friends = []
    offset = 0

    while True:
        response = vk_request(
            "friends.get",
            user_id=user_id,
            count=count,
            offset=offset
        )

        items = response.get("items", [])
        total = response.get("count", 0)

        all_friends.extend(items)

        if not items or len(all_friends) >= total or len(items) < count:
            break

        offset += len(items)
        time.sleep(REQUEST_DELAY)

    return set(all_friends)


friends_by_user = {}
api_errors = []

for i, user_id in enumerate(students["id"], start=1):
    try:
        friends_by_user[int(user_id)] = get_all_friends(int(user_id))
        print(f"[{i}/{len(students)}] ID {user_id}: "
              f"{len(friends_by_user[int(user_id)])} друзей")
    except Exception as exc:
        api_errors.append({
            "user_id": int(user_id),
            "error": str(exc)
        })
        friends_by_user[int(user_id)] = set()
        print(f"[{i}/{len(students)}] ID {user_id}: ОШИБКА — {exc}")

    time.sleep(REQUEST_DELAY)

errors_df = pd.DataFrame(api_errors)
print(f"\nОшибок API: {len(errors_df)}")
if not errors_df.empty:
    display(errors_df)

[1/4] ID 422397933: 5413 друзей
[2/4] ID 139159408: 275 друзей
[3/4] ID 830638754: 18 друзей
[4/4] ID 1098513955: 1 друзей

Ошибок API: 0


## 9. Построение графа

Ребро добавляется только между двумя студентами из исходной выборки, если первый студент видит второго в своём списке друзей VK.

Для дружбы VK граф рассматривается как **неориентированный**: наличие связи `A — B` означает одно ребро независимо от того, из чьего списка друзей оно было обнаружено.

In [23]:
student_id_set = set(students["id"].astype(int))

G = nx.Graph()

# Добавляем все вершины заранее, включая студентов без найденных связей.
for _, row in students.iterrows():
    G.add_node(
        int(row["id"]),
        name=row["name"] if row["name"] else str(row["id"]),
        group=row["group"]
    )

# Проверяем только связи внутри выборки.
for user_id, friend_ids in friends_by_user.items():
    internal_friends = student_id_set.intersection(friend_ids)

    for friend_id in internal_friends:
        if int(user_id) != int(friend_id):
            G.add_edge(int(user_id), int(friend_id))

print("Вершин:", G.number_of_nodes())
print("Рёбер:", G.number_of_edges())
print("Компонент связности:", nx.number_connected_components(G))

Вершин: 4
Рёбер: 0
Компонент связности: 4


## 10. Таблица рёбер

Эта таблица показывает, кто с кем является другом внутри учебной выборки.

In [24]:
id_to_name = dict(zip(students["id"], students["name"]))
id_to_group = dict(zip(students["id"], students["group"]))

edges = []
for u, v in G.edges():
    edges.append({
        "id_1": u,
        "name_1": id_to_name.get(u, str(u)),
        "group_1": id_to_group.get(u, ""),
        "id_2": v,
        "name_2": id_to_name.get(v, str(v)),
        "group_2": id_to_group.get(v, "")
    })

edges_df = pd.DataFrame(edges)
display(edges_df)

""


## 11. Расчёт центральностей

Используются четыре показателя:

**Центральность по степени** — доля вершин, с которыми человек непосредственно связан. Для поиска человека с максимальным количеством связей дополнительно рассматривается обычная степень `degree`.

**Центральность по посредничеству** (`betweenness`) — показывает, насколько часто вершина оказывается на кратчайших путях между другими вершинами. Высокое значение характерно для человека, который может выступать связующим звеном между частями группы.

**Центральность по близости** (`closeness`) — показывает, насколько быстро вершина достижима от остальных связанных с ней вершин по кратчайшим путям.

**Центральность собственного вектора** (`eigenvector`) — учитывает не только количество связей, но и их «качество»: связь с высокоцентральным человеком даёт больше веса. Если граф несвязный, вычисляется внутри каждой компоненты.

Если граф распадается на несколько компонент, значения closeness и eigenvector нужно интерпретировать осторожно: вершины из разных компонент недостижимы друг из друга.

In [ ]:
degree = dict(G.degree())
degree_centrality = nx.degree_centrality(G)
betweenness = nx.betweenness_centrality(G, normalized=True)
closeness = nx.closeness_centrality(G)

# Eigenvector centrality: numpy-версия устойчива к несвязным графам
try:
    eigenvector = nx.eigenvector_centrality_numpy(G)
except Exception:
    eigenvector = {n: 0.0 for n in G.nodes()}

results = students[["id", "name", "group"]].copy()
results["degree"] = results["id"].map(degree).fillna(0).astype(int)
results["degree_centrality"] = results["id"].map(degree_centrality).fillna(0)
results["betweenness_centrality"] = results["id"].map(betweenness).fillna(0)
results["closeness_centrality"] = results["id"].map(closeness).fillna(0)
results["eigenvector_centrality"] = results["id"].map(eigenvector).fillna(0)

results = results.sort_values(
    ["degree", "betweenness_centrality", "closeness_centrality"],
    ascending=False
).reset_index(drop=True)

display(
    results.style.format({
        "degree_centrality": "{:.4f}",
        "betweenness_centrality": "{:.4f}",
        "closeness_centrality": "{:.4f}",
        "eigenvector_centrality": "{:.4f}",
    })
)

## 12. Студент с наибольшим количеством связей

In [ ]:
max_degree = results["degree"].max()
leaders = results[results["degree"] == max_degree].copy()

print(f"Максимальное количество связей: {max_degree}")
display(
    leaders[[
        "id", "name", "group", "degree",
        "degree_centrality",
        "betweenness_centrality",
        "closeness_centrality",
        "eigenvector_centrality",
    ]]
)

## 13. Лидеры по трём видам центральности

In [ ]:
print("ТОП по центральности по степени:")
display(results.sort_values("degree_centrality", ascending=False).head(10))

print("ТОП по центральности по посредничеству:")
display(
    results.sort_values("betweenness_centrality", ascending=False)
           .head(10)
)

print("ТОП по центральности по близости:")
display(
    results.sort_values("closeness_centrality", ascending=False)
           .head(10)
)

print("ТОП по центральности собственного вектора:")
display(
    results.sort_values("eigenvector_centrality", ascending=False)
           .head(10)
)

## 14. Визуализация графа

Размер вершины пропорционален числу связей. Подписываются имена студентов.

In [ ]:
plt.figure(figsize=(15, 10))

if G.number_of_nodes() > 0:
    pos = nx.spring_layout(G, seed=42)

    node_sizes = [300 + 250 * G.degree(n) for n in G.nodes()]

    nx.draw_networkx(
        G,
        pos,
        node_size=node_sizes,
        with_labels=True,
        labels={n: G.nodes[n]["name"] for n in G.nodes()},
        font_size=8
    )

plt.title("Граф дружеских связей VK внутри двух учебных групп")
plt.axis("off")
plt.show()

## 15. Дополнительный анализ компонент

Полезно посмотреть, распадается ли группа на отдельные сообщества.

In [ ]:
components = sorted(nx.connected_components(G), key=len, reverse=True)

component_rows = []
for idx, component in enumerate(components, start=1):
    for node in component:
        component_rows.append({
            "component": idx,
            "component_size": len(component),
            "id": node,
            "name": id_to_name.get(node, str(node)),
            "group": id_to_group.get(node, "")
        })

components_df = pd.DataFrame(component_rows)
display(components_df)

## 16. Сохранение результатов

После выполнения анализ можно передать преподавателю как таблицы и рисунок.

In [ ]:
os.makedirs("vk_graph_results", exist_ok=True)

results.to_csv(
    "vk_graph_results/centrality_results.csv",
    index=False,
    encoding="utf-8-sig"
)

edges_df.to_csv(
    "vk_graph_results/edges.csv",
    index=False,
    encoding="utf-8-sig"
)

components_df.to_csv(
    "vk_graph_results/components.csv",
    index=False,
    encoding="utf-8-sig"
)

if not errors_df.empty:
    errors_df.to_csv(
        "vk_graph_results/api_errors.csv",
        index=False,
        encoding="utf-8-sig"
    )

with pd.ExcelWriter(
    "vk_graph_results/vk_graph_analysis.xlsx",
    engine="openpyxl"
) as writer:
    results.to_excel(writer, sheet_name="Centrality", index=False)
    edges_df.to_excel(writer, sheet_name="Edges", index=False)
    components_df.to_excel(writer, sheet_name="Components", index=False)
    if not errors_df.empty:
        errors_df.to_excel(writer, sheet_name="API_errors", index=False)

print("Результаты сохранены в папке vk_graph_results/")

## 17. Формулировка вывода для отчёта

После выполнения ячеек значения можно подставить в следующий шаблон:

> В рамках работы был сформирован граф дружеских связей студентов двух учебных групп в социальной сети VK. В качестве вершин графа рассматривались учащиеся, указанные в исходном перечне, а ребро между двумя вершинами соответствовало наличию дружеской связи между студентами. Для оценки положения вершин в графе были рассчитаны центральность по степени, центральность по посредничеству и центральность по близости. Наибольшее количество непосредственных связей имеет **[ФИО]**, количество связей — **[N]**. Максимальное значение центральности по посредничеству составляет **[X]** у **[ФИО]**, что указывает на его/её роль возможного связующего звена между другими участниками графа. Максимальная центральность по близости составляет **[Y]** у **[ФИО]**, что характеризует относительно короткое расстояние до других вершин своей компоненты связности.